In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
import os, sys
import json
import torch
import pandas as pd
import numpy as np
SAVE_PATH = '/content/drive/MyDrive/2026-1/NLP/appraisal/nlp-appraisal/data'
REPO_ROOT = os.path.dirname(SAVE_PATH)
sys.path.append(REPO_ROOT)

In [3]:
from transformers import AutoModel

# Load RoBERTa-base model to build a custom head
roberta = AutoModel.from_pretrained("roberta-base")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Checking by instantiating the model and running one forward pass on a single batch from the dataset: 

In [ ]:
from transformers import AutoTokenizer
from src.dataset import AppraisalDataset

weights_json = json.load(open('/content/drive/MyDrive/2026-1/NLP/appraisal/nlp-appraisal/data/dim_weights.json'))
target_dims = [
    'goal_relevance', 'self_responsblt', 'other_responsblt', 'chance_responsblt',
    'goal_support', 'predict_conseq', 'urgency', 'self_control', 'other_control',
    'chance_control', 'accept_conseq', 'social_norms', 'standards', 'attention', 'effort'
]
tokenizer = AutoTokenizer.from_pretrained('roberta-base')

train_dataset = AppraisalDataset(os.path.join(SAVE_PATH, 'train.csv'), tokenizer, weights_json, target_dims)
print(train_dataset[0])

Map:   0%|          | 0/4619 [00:00<?, ? examples/s]

{'input_ids': tensor([   0,  100, 8102,  708,   94, 2289,   19,   10, 1441,    2,    1,    1,
           1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,
           1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,
           1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,
           1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,
           1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,
           1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,
           1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,
           1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,
           1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,
           1,    1,    1,    1,    1,    1,    1,    1]), 'attention_mask': tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0

In [8]:
from torch.utils.data import DataLoader
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
# Grab one batch to test the dataloader

batch = next(iter(train_loader))
print(batch.keys())
print(batch['input_ids'].shape)  # Should be (batch_size, seq_len)
print(batch['attention_mask'].shape)  # Should be (batch_size, seq_len)
print(batch['labels'].shape)  # Should be (batch_size, 15) 
print(batch['weights'].shape)  # Should be (batch_size, 15)


dict_keys(['input_ids', 'attention_mask', 'labels', 'weights'])
torch.Size([16, 128])
torch.Size([16, 128])
torch.Size([16, 15])
torch.Size([16, 15])


In [9]:
# Instantiate the model and run the forward pass
from src.model import AppraisalModel
model = AppraisalModel(roberta, num_labels=15)
outputs = model(batch['input_ids'], batch['attention_mask'])
print(outputs.shape)  # Should be (batch_size, 15)
# Confirm all outputs are in [0, 1]
print(outputs.min().item(), outputs.max().item())
# Confirm loss is a scalar
from src.loss import weighted_mse_loss
loss = weighted_mse_loss(outputs, batch['labels'], batch['weights'])
print(loss.item())
# Confirm backward pass completes without error
loss.backward()
print("Backward pass completed successfully!")

torch.Size([16, 15])
0.40414950251579285 0.5916656851768494
0.020121609792113304
Backward pass completed successfully!


Freezing strategy for the first training run: 
1. Freeze all RoBERTa layers
2. Train only the linear head for 2-3 epochs

Then, unfreeze all layers and fine-tune end-to-end with a differential learning rate: 
* RoBERTa layers: lower learning rate like 2e-5
* Linear head: higher learning rate like 1e-3

NOTE: Use `AdamW` optimizer and a linear warmup scheduler for first 10% of training steps. 

In [10]:
def freeze_roberta_layers(model): 
    for param in model.base_model.parameters():
        param.requires_grad = False

def unfreeze_roberta_layers(model):
    for param in model.base_model.parameters():
        param.requires_grad = True

def confirm_trainable_status(model):
   
    base_model_params = [p.requires_grad for p in model.base_model.parameters()]
    base_model_trainable = sum(base_model_params)
    base_model_total = len(base_model_params)
    print(f"Base model trainable parameters: {base_model_trainable}/{base_model_total} ({base_model_trainable / base_model_total * 100:.2f}%)")

    linear_params = [p.requires_grad for p in model.linear.parameters()]
    linear_trainable = sum(linear_params)
    linear_total = len(linear_params)
    print(f"Linear layer trainable parameters: {linear_trainable}/{linear_total} ({linear_trainable / linear_total * 100:.2f}%)")


In [11]:
freeze_roberta_layers(model)
confirm_trainable_status(model)

Base model trainable parameters: 0/199 (0.00%)
Linear layer trainable parameters: 2/2 (100.00%)


Single training run using this structure for each epoch: 
1. Train loop: 
    a. Forward pass
    b. Weighted MSE loss 
    c. Backward
    d. Optimizer step
    e. Scheduler step
2. Validation loop: 
    a. Forward pass only (no gradient)
    b. Unweighted MSE and per-dimension RMSE on validation set
3. Log training loss, validation loss, per-dimension RMSE
4. Use early stopping to stop and save the best checkpoint

CONFIRM: Per epoch, 1) overall training loss with weighted MSE, 2) overall validation loss with unweighted MSE, 3) per-dimension validation RMSE for all 15 dimensions are recorded

Batch size: 16
Max epochs: 10 (with early stopping)